# 1. Set up enviroment and data

## 1.1. Import libraries

In [1]:
import pandas as pd
import datetime as dt
import numpy as np
import seaborn as sns

## 1.2. Import data to dataframes

In [2]:
data1=pd.read_csv('./data/CusData.csv')
data1.drop(columns=["Unnamed: 0"], inplace=True)
data1.head()

In [ ]:
data2=pd.read_csv('./data/TransactionData.csv')
data2.head()

In [ ]:
data=pd.DataFrame(data2)
data3=pd.DataFrame(data2)

In [ ]:
data.drop(columns=["Unnamed: 0",'FOB_RACING_TURNOVER',	'FOB_SPORT_TURNOVER', 'PARI_RACING_TURNOVER', 'PARI_SPORT_TURNOVER', 'DIVIDENDS_PAID',	'GROSS_MARGIN',	'TICKETS'], inplace=True)
data.head()

# 2. RFM Calculation

In [ ]:
# Tìm ngày lớn nhất
data['DATE_DIM'] = pd.to_datetime(data['DATE_DIM'])
max_date = data['DATE_DIM'].max()
current_date = max_date + pd.Timedelta(days=1)
print("Ngày hiện tại:", current_date)

In [ ]:
# Tính toán Recency bằng cách tính số ngày kể từ ngày gần nhất
data['recency'] = (current_date - data['DATE_DIM']).dt.days
data.head()

In [ ]:
# Tính toán Monetary
data['money'] = data['TOTAL_TURNOVER']
data.head()

In [ ]:
# Tính toán Frequency bằng cách đếm số lần xuất hiện của mỗi BET_ACCOUNT_NUM_HASH
freq_data = data.groupby('BET_ACCOUNT_NUM_HASH').size().reset_index(name='freq')

In [ ]:
# Kết hợp dữ liệu recency, monetary và frequency lại với nhau
rfm_data = data.groupby('BET_ACCOUNT_NUM_HASH').agg({'recency': 'min', 'money': 'sum'}).reset_index()
rfm_data = pd.merge(rfm_data, freq_data, on='BET_ACCOUNT_NUM_HASH')
rfm_data[['recency', 'money', 'freq']] = rfm_data[['recency', 'money', 'freq']].astype(float)
# Hiển thị dữ liệu RFM
rfm_data.head()

In [ ]:
rfm_data.describe()

# 3. RFM score

In [ ]:
rfm_c=rfm_data

In [ ]:
rfm_c['r_c'] = pd.qcut(rfm_c['recency'], 5, ['5','4','3','2','1'])
rfm_c['f_c'] = pd.qcut(rfm_c['freq'], 5, ['1','2','3','4','5'])
rfm_c['m_c'] = pd.qcut(rfm_c['money'], 5, ['1','2','3','4','5'])
rfm_c.head()

In [ ]:
rfm_c['RFM_Score'] = rfm_c.r_c.astype(str)+ rfm_c.f_c.astype(str) + rfm_c.m_c.astype(str)
rfm_c.head()

In [ ]:
# Specify the file path where you want to save the CSV file
file_path = "./data/rfm_c.csv"
# Export merged data to CSV
rfm_c.to_csv(file_path, index=False)

In [ ]:
# Phân cụm theo recency, frequency và monetary và lưu trữ các cạnh bin
r_bins = pd.qcut(rfm_c['recency'], 5).unique().sort_values()
f_bins = pd.qcut(rfm_c['freq'], 5).unique().sort_values()
m_bins = pd.qcut(rfm_c['money'], 5).unique().sort_values()

# In ra các cạnh bin cho từng chỉ số RFM
print("Recency bin edges:", r_bins)
print("Frequency bin edges:", f_bins)
print("Monetary bin edges:", m_bins)